In [1]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import IntSlider, fixed, interact

%matplotlib inline

from config import IMAGE_DATA_DIR
from kuwahara_filters.classic import classic_kuwahara, classic_kuwahara_flawed

In [2]:
def load_and_resize(image_path, max_side=800):
    img = cv2.imread(image_path, cv2.IMREAD_COLOR_RGB)
    h, w = img.shape[:2]

    if max(h, w) <= max_side:
        return img

    scale = max_side / float(max(h, w))
    new_w = int(w * scale)
    new_h = int(h * scale)

    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

In [3]:
def add_gaussian_noise(image, sigma):
    if sigma == 0:
        return image.copy()

    noise = np.random.normal(0, sigma, image.shape)
    noisy_image = image.astype(np.float32) + noise

    return np.clip(noisy_image, 0, 255).astype(np.uint8)

In [4]:
def update_kuwahara(image, noise_sigma, kuwahara_algorithm, **kuwahara_params):
    noisy_image = add_gaussian_noise(image, sigma=noise_sigma)
    denoised_image = kuwahara_algorithm(noisy_image, **kuwahara_params)

    fig, axes = plt.subplots(1, 3, figsize=(15, 7))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(noisy_image)
    axes[1].set_title(f"Noisy (sigma={noise_sigma})")
    axes[1].axis("off")

    axes[2].imshow(denoised_image)
    params = ",".join(
        "=".join(map(str, pair)) for pair in kuwahara_params.items()
    )
    axes[2].set_title(f"Classic ({params})")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

In [25]:
original_image = load_and_resize(IMAGE_DATA_DIR / "landscape.png", max_side=600)

In [26]:
interact(
    update_kuwahara,
    image=fixed(original_image),
    kuwahara_algorithm=fixed(classic_kuwahara),
    noise_sigma=IntSlider(min=0, max=100, step=5, value=10),
    radius=IntSlider(min=1, max=10, step=1, value=1),
)

interactive(children=(IntSlider(value=10, description='noise_sigma', step=5), IntSlider(value=1, description='…

<function __main__.update_kuwahara(image, noise_sigma, kuwahara_algorithm, **kuwahara_params)>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 6))

correct = classic_kuwahara(original_image, radius=5)
flawed = classic_kuwahara_flawed(original_image, radius=5)
diff = diff = cv2.absdiff(correct, flawed)

axes[0].imshow(correct)
axes[0].set_title("Correct implementation")

axes[1].imshow(flawed)
axes[1].set_title("Flawed implementation")

axes[2].imshow(diff)
axes[2].set_title("Image difference")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
y, x = 250, 300
h, w = 60, 60

crop_correct = correct[y : y + h, x : x + w]
crop_flawed = flawed[y : y + h, x : x + w]

img_with_box = original_image.copy()
cv2.rectangle(img_with_box, (x, y), (x + w, y + h), (255, 0, 0), 2)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_with_box)
axes[0].set_title("Original Image")

axes[1].imshow(crop_correct)
axes[1].set_title("Correct implementation (Zoom)")

axes[2].imshow(crop_flawed)
axes[2].set_title("Flawed implementation (Zoom)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
cv2.imwrite("fig_full_ref.png", cv2.cvtColor(img_with_box, cv2.COLOR_RGB2BGR))
cv2.imwrite(
    "fig_crop_correct.png", cv2.cvtColor(crop_correct, cv2.COLOR_RGB2BGR)
)
cv2.imwrite("fig_crop_flawed.png", cv2.cvtColor(crop_flawed, cv2.COLOR_RGB2BGR))